<a href="https://colab.research.google.com/github/kalyanisawkar/movie-recommendation-system-analysis/blob/main/movie_Recommendator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 55.0 MB/s eta 0:00:00


In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
 %%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

st.set_page_config(page_title="🎬 Movie Recommender", layout="wide")
st.title("🎬 Movie Recommendation System")
st.markdown("---")

# ── Load Dataset ──────────────────────────────────────────
@st.cache_data
def load_data():
    URL = "https://raw.githubusercontent.com/rashida048/Some-NLP-Projects/master/movie_dataset.csv"
    df = pd.read_csv(URL)
    return df

with st.spinner("Loading dataset..."):
    df = load_data()
st.success(f"✅ Loaded {len(df)} movies!")

# ── Feature Engineering ───────────────────────────────────
FEATURES = ["keywords", "cast", "genres", "director", "overview"]
for col in FEATURES:
    if col in df.columns:
        df[col] = df[col].fillna("").str.lower().str.replace(",", " ")

if "director" in df.columns:
    df["director_w"] = df["director"].apply(lambda x: " ".join([str(x)] * 3))
else:
    df["director_w"] = ""

existing = [c for c in ["keywords", "cast", "genres", "director_w", "overview"] if c in df.columns]
df["soup"] = df[existing].apply(lambda row: " ".join(row.values.astype(str)), axis=1)
df["title"] = df["title"].fillna("Unknown")

# ── Build Models ──────────────────────────────────────────
@st.cache_resource
def build_models(soup_series):
    cv = CountVectorizer(stop_words="english", max_features=10000, ngram_range=(1, 2))
    cv_matrix = cv.fit_transform(soup_series)
    cv_sim = cosine_similarity(cv_matrix, cv_matrix)

    tfidf = TfidfVectorizer(stop_words="english", max_features=10000)
    tfidf_matrix = tfidf.fit_transform(soup_series)
    tfidf_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    return cv_sim, tfidf_sim

with st.spinner("Building models..."):
    cv_sim, tfidf_sim = build_models(df["soup"])

title_to_idx = pd.Series(df.index, index=df["title"].str.lower()).drop_duplicates()

scaler = MinMaxScaler()
df["norm_vote"] = scaler.fit_transform(df[["vote_average"]].fillna(0))
df["norm_pop"]  = scaler.fit_transform(df[["vote_count"]].fillna(0).apply(np.log1p))

# ── Recommendation Functions ──────────────────────────────
def get_recs(title, sim_matrix, top_n=10):
    idx = title_to_idx[title.lower()]
    scores = sorted(enumerate(sim_matrix[idx]), key=lambda x: x[1], reverse=True)[1:top_n+1]
    indices = [i[0] for i in scores]
    vals    = [round(i[1], 4) for i in scores]
    cols = ["title"] + [c for c in ["genres", "director", "vote_average", "release_date"] if c in df.columns]
    result = df.iloc[indices][cols].copy()
    result["score"] = vals
    result.index = range(1, len(result)+1)
    return result

def hybrid_recs(title, top_n=10, w_sim=0.6, w_vote=0.3, w_pop=0.1):
    idx = title_to_idx[title.lower()]
    sim_arr  = cv_sim[idx].copy()
    norm_sim = MinMaxScaler().fit_transform(sim_arr.reshape(-1, 1)).flatten()
    hybrid   = w_sim * norm_sim + w_vote * df["norm_vote"].values + w_pop * df["norm_pop"].values
    scores   = sorted(enumerate(hybrid), key=lambda x: x[1], reverse=True)
    scores   = [s for s in scores if s[0] != idx][:top_n]
    indices  = [i[0] for i in scores]
    vals     = [round(i[1], 4) for i in scores]
    cols = ["title"] + [c for c in ["genres", "director", "vote_average", "vote_count"] if c in df.columns]
    result = df.iloc[indices][cols].copy()
    result["hybrid_score"] = vals
    result.index = range(1, len(result)+1)
    return result

# ════════════════════════════════════════════════════════
#  SIDEBAR
# ════════════════════════════════════════════════════════
st.sidebar.header("⚙️ Settings")
model_choice = st.sidebar.radio("Choose Model", ["CountVectorizer", "TF-IDF", "Hybrid"])
top_n = st.sidebar.slider("Number of Recommendations", 5, 20, 10)

if model_choice == "Hybrid":
    w_sim  = st.sidebar.slider("Weight: Similarity",   0.0, 1.0, 0.6)
    w_vote = st.sidebar.slider("Weight: Vote Average", 0.0, 1.0, 0.3)
    w_pop  = st.sidebar.slider("Weight: Popularity",   0.0, 1.0, 0.1)

movie_list = sorted(df["title"].dropna().unique().tolist())
selected   = st.sidebar.selectbox("🎥 Select a Movie", movie_list)
run_btn    = st.sidebar.button("🔍 Get Recommendations")

# ════════════════════════════════════════════════════════
#  TAB LAYOUT
# ════════════════════════════════════════════════════════
tab1, tab2, tab3 = st.tabs(["🎯 Recommendations", "📊 Dashboard", "🔥 Similarity Heatmap"])

# ════════════════════════════════════════════════════════
#  TAB 1 — RECOMMENDATIONS
# ════════════════════════════════════════════════════════
with tab1:
    st.subheader(f"🎬 Recommendations for: **{selected}**")

    if run_btn:
        try:
            if model_choice == "CountVectorizer":
                recs = get_recs(selected, cv_sim, top_n)
                score_col = "score"
            elif model_choice == "TF-IDF":
                recs = get_recs(selected, tfidf_sim, top_n)
                score_col = "score"
            else:
                recs = hybrid_recs(selected, top_n, w_sim, w_vote, w_pop)
                score_col = "hybrid_score"

            # Table
            st.dataframe(recs, use_container_width=True)

            # Bar Chart
            fig, ax = plt.subplots(figsize=(9, 5))
            colors = sns.color_palette("viridis", len(recs))
            ax.barh(recs["title"][::-1], recs[score_col][::-1], color=colors)
            for i, (val, name) in enumerate(zip(recs[score_col][::-1], recs["title"][::-1])):
                ax.text(val + 0.001, i, f"{val:.3f}", va="center", fontsize=8)
            ax.set_xlabel("Score")
            ax.set_title(f"Top {top_n} Recommendations — {model_choice}")
            plt.tight_layout()
            st.pyplot(fig)

            # CountVec vs TF-IDF Comparison
            st.subheader("📐 CountVectorizer vs TF-IDF Score Comparison")
            cv_r   = get_recs(selected, cv_sim, top_n)
            tfidf_r= get_recs(selected, tfidf_sim, top_n)
            common = pd.merge(
                cv_r[["title", "score"]].rename(columns={"score": "CountVec"}),
                tfidf_r[["title", "score"]].rename(columns={"score": "TF-IDF"}),
                on="title"
            )
            if not common.empty:
                x  = np.arange(len(common))
                w  = 0.35
                fig2, ax2 = plt.subplots(figsize=(10, 5))
                ax2.bar(x - w/2, common["CountVec"], w, label="CountVec", color="steelblue")
                ax2.bar(x + w/2, common["TF-IDF"],   w, label="TF-IDF",   color="coral")
                ax2.set_xticks(x)
                ax2.set_xticklabels(common["title"], rotation=40, ha="right", fontsize=8)
                ax2.set_ylabel("Similarity Score")
                ax2.set_title(f"Model Comparison for «{selected}»")
                ax2.legend()
                plt.tight_layout()
                st.pyplot(fig2)

        except Exception as e:
            st.error(f"❌ Error: {e}")
    else:
        st.info("👈 Select a movie and click **Get Recommendations** in the sidebar")

# ════════════════════════════════════════════════════════
#  TAB 2 — DASHBOARD
# ════════════════════════════════════════════════════════
with tab2:
    st.subheader("📊 Dataset Overview")

    # Metrics Row
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("🎬 Total Movies",     len(df))
    m2.metric("⭐ Avg Rating",        f"{df['vote_average'].mean():.2f}")
    m3.metric("🎥 Unique Directors",  df["director"].nunique())
    m4.metric("🗂️ Genres Available",  df["genres"].nunique())

    st.markdown("---")

    # Row 1
    col1, col2 = st.columns(2)

    with col1:
        st.markdown("**⭐ Vote Average Distribution**")
        fig, ax = plt.subplots(figsize=(6, 4))
        va = df["vote_average"].dropna()
        ax.hist(va, bins=25, color="steelblue", edgecolor="white")
        ax.axvline(va.mean(), color="red", linestyle="--", label=f"Mean: {va.mean():.2f}")
        ax.set_xlabel("Vote Average")
        ax.set_ylabel("Count")
        ax.legend()
        ax.set_title("Distribution of Movie Ratings")
        st.pyplot(fig)

    with col2:
        st.markdown("**🎭 Top 10 Genres**")
        all_g   = " ".join(df["genres"].dropna()).split()
        top_g   = Counter([g for g in all_g if len(g) > 2]).most_common(10)
        labels, vals = zip(*top_g)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(labels, vals, color=sns.color_palette("Set2", len(labels)))
        ax.set_title("Most Common Genre Tags")
        ax.set_xlabel("Genre")
        ax.set_ylabel("Frequency")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        st.pyplot(fig)

    # Row 2
    col3, col4 = st.columns(2)

    with col3:
        st.markdown("**🎬 Top 10 Directors by Movie Count**")
        top_dirs = df["director"].value_counts().head(10)
        fig, ax  = plt.subplots(figsize=(6, 4))
        ax.barh(top_dirs.index[::-1], top_dirs.values[::-1],
                color=sns.color_palette("magma", 10))
        ax.set_xlabel("Number of Movies")
        ax.set_title("Most Prolific Directors")
        plt.tight_layout()
        st.pyplot(fig)

    with col4:
        st.markdown("**📅 Movies Released per Year (Top 20 Years)**")
        if "release_date" in df.columns:
            df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
            year_counts = df["year"].value_counts().sort_index().tail(30)
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.plot(year_counts.index, year_counts.values,
                    color="darkorange", linewidth=2, marker="o", markersize=3)
            ax.set_xlabel("Year")
            ax.set_ylabel("Movies Released")
            ax.set_title("Movies Over Time")
            plt.tight_layout()
            st.pyplot(fig)

# ════════════════════════════════════════════════════════
#  TAB 3 — HEATMAP
# ════════════════════════════════════════════════════════
with tab3:
    st.subheader("🔥 Cosine Similarity Heatmap")
    n_movies = st.slider("Number of movies to compare", 10, 30, 15)

    top_movies = df["title"].head(n_movies).tolist()
    idx_list   = [title_to_idx[t.lower()] for t in top_movies if t.lower() in title_to_idx]
    sub_sim    = cv_sim[np.ix_(idx_list, idx_list)]
    labels     = [top_movies[i] for i in range(len(idx_list))]

    fig, ax = plt.subplots(figsize=(12, 9))
    sns.heatmap(sub_sim, xticklabels=labels, yticklabels=labels,
                cmap="YlOrRd", linewidths=0.3, ax=ax,
                annot=True if n_movies <= 15 else False, fmt=".2f", annot_kws={"fontsize": 7})
    ax.set_title(f"Cosine Similarity — Top {n_movies} Movies")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout()
    st.pyplot(fig)

Writing app.py


In [ ]:
# Kill old tunnels and restart
from pyngrok import ngrok
import subprocess, time

ngrok.kill()
time.sleep(2)

ngrok.set_auth_token("3AhIltxHo9ZD39vGSkl3CORHntr_4veErxySVwLp5hXY1qUXZ")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])
time.sleep(3)

public_url = ngrok.connect(8501)
print("🚀 Your Streamlit App is live at:", public_url)


🚀 Your Streamlit App is live at: NgrokTunnel: "https://hermitically-unmoldable-shella.ngrok-free.dev" -> "http://localhost:8501"
